# Diffusion Map on XY configurations

This notebook loads XY-model configurations saved via `build_xy_dataset` in `xy_sim.py`, builds a feature matrix, and runs a diffusion map embedding to visualize the temperature manifold.

In [ ]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# Path to the compressed dataset produced by build_xy_dataset
DATA_PATH = Path('xy_dataset_wolff.npz')
METADATA_PATH = Path('xy_dataset_wolff.metadata.json')

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset {DATA_PATH} not found. Update DATA_PATH to point to your file.")
if not METADATA_PATH.exists():
    raise FileNotFoundError(f"Metadata file {METADATA_PATH} not found.")

with np.load(DATA_PATH) as npz:
    configurations = npz['configurations']

with open(METADATA_PATH, 'r', encoding='utf-8') as f:
    metadata = json.load(f)

num_samples, num_sites = configurations.shape
print(f"Loaded {num_samples} samples with {num_sites} lattice sites each.")


FileNotFoundError: Metadata file xy_dataset_wolff.npz.metadata.json not found.

In [ ]:
# Convert flattened angles (radians) into (cos, sin) features per site
angles = configurations.astype(np.float64)
features = np.stack([np.cos(angles), np.sin(angles)], axis=-1)
features = features.reshape(num_samples, -1)
print(f"Feature matrix shape: {features.shape}")

# Extract temperatures for coloring
temperatures = np.array([entry['temperature'] for entry in metadata], dtype=np.float64)
assert temperatures.shape[0] == num_samples


In [ ]:
from sklearn.metrics import pairwise_distances
from sklearn.preprocessing import normalize

# Diffusion map helper

def diffusion_map(data, epsilon=None, n_components=3):
    # Pairwise Euclidean distances
    dists = pairwise_distances(data, metric='euclidean')
    if epsilon is None:
        # median of squared distances as heuristic bandwidth
        eps = np.median(dists ** 2)
        epsilon = max(eps, 1e-12)
    kernel = np.exp(-(dists ** 2) / epsilon)
    # Normalize to Markov matrix
    row_sums = kernel.sum(axis=1, keepdims=True)
    P = kernel / row_sums
    # Symmetrize for stable eigendecomposition
    D_half = np.sqrt(row_sums)
    A = kernel / (D_half @ D_half.T)
    # Eigen decomposition
    eigvals, eigvecs = np.linalg.eigh(A)
    idx = np.argsort(eigvals)[::-1]
    eigvals = eigvals[idx]
    eigvecs = eigvecs[:, idx]
    # Skip the first eigenvector (constant mode)
    diffusion_coords = eigvecs[:, 1:n_components+1]
    diffusion_vals = eigvals[1:n_components+1]
    # Normalize coordinates by eigenvalues (diffusion map convention)
    diffusion_coords = diffusion_coords * diffusion_vals
    return diffusion_coords, diffusion_vals, epsilon

embeddings, lambdas, eps = diffusion_map(features, n_components=3)
print("Eigenvalues:", lambdas)
print("Epsilon used:", eps)


In [ ]:
# Plot first two diffusion coordinates colored by temperature
plt.figure(figsize=(6, 5))
sc = plt.scatter(embeddings[:, 0], embeddings[:, 1], c=temperatures, cmap='plasma', s=30, edgecolor='none')
plt.xlabel('Diffusion coord 1')
plt.ylabel('Diffusion coord 2')
plt.title('Diffusion map embedding of XY configurations')
cb = plt.colorbar(sc)
cb.set_label('Temperature')
plt.grid(alpha=0.3)
plt.show()


In [ ]:
# 3D visualization of the first three diffusion coordinates
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

fig = plt.figure(figsize=(7, 6))
ax = fig.add_subplot(111, projection='3d')
scatter = ax.scatter(
    embeddings[:, 0],
    embeddings[:, 1],
    embeddings[:, 2],
    c=temperatures,
    cmap='plasma',
    s=35,
    edgecolor='none'
)
ax.set_xlabel('Coord 1')
ax.set_ylabel('Coord 2')
ax.set_zlabel('Coord 3')
ax.set_title('Diffusion map embedding (3D)')
fig.colorbar(scatter, ax=ax, shrink=0.7, label='Temperature')
plt.show()


In [ ]:
# Optional: inspect embeddings vs temperature ordering
order = np.argsort(temperatures)
plt.figure(figsize=(6, 4))
plt.plot(temperatures[order], embeddings[order, 0], label='Coord 1')
plt.plot(temperatures[order], embeddings[order, 1], label='Coord 2')
plt.xlabel('Temperature (sorted)')
plt.ylabel('Diffusion coordinate value')
plt.title('Diffusion coordinates across temperature')
plt.legend()
plt.grid(alpha=0.3)
plt.show()


## Interactive exploration

Use the widgets below to explore the diffusion-map coordinates interactively. You can filter by temperature range, pick which diffusion coordinates to examine, adjust the point size, or toggle a 3D view.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

coord_options = {
    'Coord 1 vs 2': (0, 1),
    'Coord 1 vs 3': (0, 2),
    'Coord 2 vs 3': (1, 2),
}

temp_min = float(temperatures.min())
temp_max = float(temperatures.max())

temp_slider = widgets.FloatRangeSlider(
    value=(temp_min, temp_max),
    min=temp_min,
    max=temp_max,
    step=0.01,
    description='Temp range',
    continuous_update=False,
    layout=widgets.Layout(width='70%')
)

coord_dropdown = widgets.Dropdown(
    options=[(label, pair) for label, pair in coord_options.items()],
    value=(0, 1),
    description='2D axes'
)

size_slider = widgets.IntSlider(value=35, min=10, max=120, step=5, description='Point size')
show_3d_checkbox = widgets.Checkbox(value=False, description='3D scatter')

@widgets.interact(
    coord_pair=coord_dropdown,
    temp_range=temp_slider,
    point_size=size_slider,
    show_3d=show_3d_checkbox,
)
def interactive_diffusion(coord_pair, temp_range, point_size, show_3d):
    mask = (temperatures >= temp_range[0]) & (temperatures <= temp_range[1])
    if not np.any(mask):
        print('No samples in the selected temperature window.')
        return

    if show_3d:
        fig = plt.figure(figsize=(7, 6))
        ax = fig.add_subplot(111, projection='3d')
        coords3d = embeddings[mask, :3]
        sc = ax.scatter(
            coords3d[:, 0], coords3d[:, 1], coords3d[:, 2],
            c=temperatures[mask], cmap='plasma', s=point_size, edgecolor='none'
        )
        ax.set_xlabel('Coord 1')
        ax.set_ylabel('Coord 2')
        ax.set_zlabel('Coord 3')
        ax.set_title('Diffusion map embedding (interactive 3D)')
        fig.colorbar(sc, ax=ax, shrink=0.7, label='Temperature')
        plt.show()
        return

    x_idx, y_idx = coord_pair
    fig, ax = plt.subplots(figsize=(6, 5))
    sc = ax.scatter(
        embeddings[mask, x_idx],
        embeddings[mask, y_idx],
        c=temperatures[mask],
        cmap='plasma',
        s=point_size,
        edgecolor='none'
    )
    ax.set_xlabel(f'Diffusion coord {x_idx + 1}')
    ax.set_ylabel(f'Diffusion coord {y_idx + 1}')
    ax.set_title('Diffusion map (interactive 2D)')
    fig.colorbar(sc, ax=ax, label='Temperature')
    ax.grid(alpha=0.3)
    plt.show()
